### Issues
- order_purchase_timestamp is the wrong data type(string) and also the time stamp format is inconsitent
- order_approved_at the column name is inconsistent with that of it peers 
- order_delivered_timestamp has null values
### Solution
- order_purchase the data type would be changed to time stamp and the data format would be transforerd for consitency
- order_approved_at the column name would be changed to order_approved_timestamp 
- order_delivered_time the null values would be left as is


In [0]:
df = spark.table('ecommerce.bronze.orders')
df.show()

+----------+-----------+------------+------------------------+-------------------+-------------------------+
|  order_id|customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_timestamp|
+----------+-----------+------------+------------------------+-------------------+-------------------------+
|ORD0000000| CUST000980|   delivered|     2024-04-20 16:34:19|2024-04-20 22:34:19|      2024-05-09 22:34:19|
|ORD0000001| CUST001017|   delivered|     2023-02-02 10:06:29|2023-02-02 17:06:29|      2023-02-05 17:06:29|
|ORD0000002| CUST001600|   delivered|     2023-12-02 04:01:59|2023-12-03 11:01:59|      2023-12-14 11:01:59|
|ORD0000003| CUST000053|    invoiced|     2023-01-17 15:43:52|2023-01-17 19:43:52|                     NULL|
|ORD0000004| CUST001761|  processing|     2024-02-19 21:56:46|2024-02-21 12:56:46|                     NULL|
|ORD0000005| CUST000106|   delivered|     2023-01-15 01:18:45|2023-01-16 13:18:45|      2023-02-02 13:18:45|
|ORD0000006| CUST00

In [0]:
from pyspark.sql.functions import col, coalesce, try_to_timestamp, lit

df = df.withColumn(
    "order_purchase_timestamp",
    coalesce(
        try_to_timestamp(
            col("order_purchase_timestamp"),
            lit("yyyy-MM-dd HH:mm:ss")
        ),
        try_to_timestamp(
            col("order_purchase_timestamp"),
            lit("dd/MM/yyyy HH:mm")
        )
    )
)

In [0]:
display(df)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_timestamp
ORD0000000,CUST000980,delivered,2024-04-20T16:34:19.000Z,2024-04-20T22:34:19.000Z,2024-05-09T22:34:19.000Z
ORD0000001,CUST001017,delivered,2023-02-02T10:06:29.000Z,2023-02-02T17:06:29.000Z,2023-02-05T17:06:29.000Z
ORD0000002,CUST001600,delivered,2023-12-02T04:01:59.000Z,2023-12-03T11:01:59.000Z,2023-12-14T11:01:59.000Z
ORD0000003,CUST000053,invoiced,2023-01-17T15:43:52.000Z,2023-01-17T19:43:52.000Z,null
ORD0000004,CUST001761,processing,2024-02-19T21:56:46.000Z,2024-02-21T12:56:46.000Z,null
ORD0000005,CUST000106,delivered,2023-01-15T01:18:45.000Z,2023-01-16T13:18:45.000Z,2023-02-02T13:18:45.000Z
ORD0000006,CUST001125,delivered,2023-04-08T06:35:39.000Z,2023-04-09T10:35:39.000Z,2023-04-24T10:35:39.000Z
ORD0000007,CUST000720,delivered,2024-11-27T20:01:37.000Z,2024-11-29T12:01:37.000Z,2024-12-12T12:01:37.000Z
ORD0000008,CUST000188,delivered,2023-02-14T23:45:13.000Z,2023-02-15T15:45:13.000Z,2023-02-24T15:45:13.000Z
ORD0000009,CUST000093,delivered,2024-08-03T13:57:24.000Z,2024-08-05T00:57:24.000Z,2024-08-25T00:57:24.000Z


In [0]:
df = df.withColumnRenamed("order_approved_at", "order_approved_timestamp")
df.columns

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_timestamp',
 'order_delivered_timestamp']

In [0]:
df.write.format('Delta').mode('overwrite').saveAsTable('ecommerce.silver.orders')
display(spark.table('ecommerce.silver.orders'))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_timestamp,order_delivered_timestamp
ORD0000000,CUST000980,delivered,2024-04-20T16:34:19.000Z,2024-04-20T22:34:19.000Z,2024-05-09T22:34:19.000Z
ORD0000001,CUST001017,delivered,2023-02-02T10:06:29.000Z,2023-02-02T17:06:29.000Z,2023-02-05T17:06:29.000Z
ORD0000002,CUST001600,delivered,2023-12-02T04:01:59.000Z,2023-12-03T11:01:59.000Z,2023-12-14T11:01:59.000Z
ORD0000003,CUST000053,invoiced,2023-01-17T15:43:52.000Z,2023-01-17T19:43:52.000Z,null
ORD0000004,CUST001761,processing,2024-02-19T21:56:46.000Z,2024-02-21T12:56:46.000Z,null
ORD0000005,CUST000106,delivered,2023-01-15T01:18:45.000Z,2023-01-16T13:18:45.000Z,2023-02-02T13:18:45.000Z
ORD0000006,CUST001125,delivered,2023-04-08T06:35:39.000Z,2023-04-09T10:35:39.000Z,2023-04-24T10:35:39.000Z
ORD0000007,CUST000720,delivered,2024-11-27T20:01:37.000Z,2024-11-29T12:01:37.000Z,2024-12-12T12:01:37.000Z
ORD0000008,CUST000188,delivered,2023-02-14T23:45:13.000Z,2023-02-15T15:45:13.000Z,2023-02-24T15:45:13.000Z
ORD0000009,CUST000093,delivered,2024-08-03T13:57:24.000Z,2024-08-05T00:57:24.000Z,2024-08-25T00:57:24.000Z
